# MDLM-Style Masked Diffusion on 32x32 Mazes

This notebook adapts the absorbing-state masked diffusion model from MDLM to fixed-size binary maze images. The forward process replaces cells with a dedicated `[MASK]` token. A timestep-conditioned U-Net predicts the original wall/passage class at every position, and the reverse process progressively unmasks the grid.

The implementation uses the MDLM substitution parameterization and its discrete-time masked cross-entropy objective. It keeps the project's U-Net backbone rather than the transformer used in the language-model experiments. Run cells manually; training may take a while.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

import importlib
import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import clear_output, display
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from plotly.subplots import make_subplots

from _shared.maze_data import generate_dataset, solvability_rate
import _shared.model as model_module

importlib.reload(model_module)
from _shared.model import Denoiser

SEED = 7
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Project: {PROJECT_ROOT} | device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} | CUDA runtime: {torch.version.cuda}")
else:
    print(
        f"CUDA unavailable to PyTorch (torch.version.cuda={torch.version.cuda}). Run uv sync and select the project .venv kernel."
    )

Project: c:\Users\shich\Src\discrete_diffusion | device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU | CUDA runtime: 12.8


## 1. Synthetic Dataset

Randomized DFS generates perfect mazes. `0` means wall; `1` means passage.

In [2]:
N_MAZES = 12_000
VAL_FRACTION = 0.1
mazes = generate_dataset(N_MAZES, seed=SEED)
n_val = int(N_MAZES * VAL_FRACTION)
train_mazes = torch.from_numpy(mazes[:-n_val].copy()).long()
val_mazes = torch.from_numpy(mazes[-n_val:].copy()).long()
print(
    "all:", mazes.shape, "| train:", len(train_mazes), "| validation:", len(val_mazes)
)
print(f"Generated-maze solvability: {solvability_rate(mazes):.1%}")

fig = go.Figure(
    go.Heatmap(
        z=mazes[0],
        zmin=0,
        zmax=1,
        colorscale=[[0, "black"], [1, "white"]],
        showscale=False,
    )
)
fig.update_layout(
    title="Synthetic maze example",
    width=420,
    height=420,
    yaxis=dict(autorange="reversed", scaleanchor="x"),
)
fig.show()

all: (12000, 32, 32) | train: 10800 | validation: 1200
Generated-maze solvability: 100.0%


## 2. Forward Masking Process

Use an absorbing `[MASK]` state. At time `t`, each original pixel is retained with probability `alpha_t` and replaced by `[MASK]` with probability `1 - alpha_t`. This notebook uses the linear retention schedule `alpha_t = 1 - t/T`; at `t=T` the entire grid is masked.

The visualization uses black for wall, white for passage, and gray for `[MASK]`. The image remains categorical throughout.

In [3]:
NUM_CLASSES = 2
MASK_TOKEN = 2
NUM_INPUT_STATES = 3
DIFFUSION_STEPS = 100
alpha = torch.linspace(1.0, 0.0, DIFFUSION_STEPS + 1, device=DEVICE)


def q_sample(x0, timesteps):
    alpha_t = alpha[timesteps]
    masked = torch.rand(x0.shape, device=x0.device) >= alpha_t[:, None, None]
    return torch.where(masked, MASK_TOKEN, x0)


def one_hot_states(x):
    return F.one_hot(x.long(), num_classes=NUM_INPUT_STATES).permute(0, 3, 1, 2).float()


preview_maze = torch.from_numpy(mazes[0].copy()).long().unsqueeze(0).to(DEVICE)
preview_timesteps = [0, 1, 2, 4, 7, 11, 16, 22, 30, 39, 49, 59, 69, 79, 89, 94, 97, 100]
mask_colors = [
    [0.0, "black"],
    [0.249, "black"],
    [0.25, "white"],
    [0.749, "white"],
    [0.75, "#999999"],
    [1.0, "#999999"],
]
with torch.no_grad():
    forward_frames = []
    for t in preview_timesteps:
        timestep = torch.full((1,), t, device=DEVICE, dtype=torch.long)
        forward_frames.append(q_sample(preview_maze, timestep)[0].cpu().numpy())

forward_fig = make_subplots(
    rows=2,
    cols=9,
    subplot_titles=[f"t={t}" for t in preview_timesteps],
)
for index, frame in enumerate(forward_frames):
    forward_fig.add_trace(
        go.Heatmap(
            z=frame,
            zmin=0,
            zmax=2,
            colorscale=mask_colors,
            showscale=False,
        ),
        row=index // 9 + 1,
        col=index % 9 + 1,
    )
forward_fig.update_layout(
    title="MDLM forward masking: black=wall, white=passage, gray=[MASK]",
    width=1600,
    height=650,
)
forward_fig.update_xaxes(showticklabels=False)
forward_fig.update_yaxes(showticklabels=False, autorange="reversed")
forward_fig.show()

## 3. Model Architecture

The input has three channels: wall, passage, and `[MASK]`. The output head has two channels and predicts only the clean wall/passage classes. The shared U-Net keeps encoder widths `64 -> 128 -> 256`, residual convolution blocks, and a `128`-dimensional timestep embedding. `output_dim=2` separates the input-state count from the predicted data vocabulary.

In [4]:
BASE_CHANNELS = 64
TIME_DIM = 128

model = Denoiser(
    embedding_dim=NUM_INPUT_STATES,
    output_dim=NUM_CLASSES,
    base_channels=BASE_CHANNELS,
    time_dim=TIME_DIM,
).to(DEVICE)
print(model)
print(
    f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)

Denoiser(
  (time_mlp): Sequential(
    (0): Linear(in_features=128, out_features=512, bias=True)
    (1): SiLU()
    (2): Linear(in_features=512, out_features=128, bias=True)
  )
  (input): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (enc32): ResBlock(
    (norm1): GroupNorm(8, 64, eps=1e-05, affine=True)
    (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (time_projection): Linear(in_features=128, out_features=64, bias=True)
    (norm2): GroupNorm(8, 64, eps=1e-05, affine=True)
    (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (skip): Identity()
  )
  (down16): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
  (enc16): ResBlock(
    (norm1): GroupNorm(8, 128, eps=1e-05, affine=True)
    (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (time_projection): Linear(in_features=128, out_features=128, bias=True)
    (norm2): GroupNorm(8, 128, eps=1e-05, a

## 4. MDLM Training

Sample a discrete timestep and mask pixels independently using its retention probability. The network predicts `p_theta(x_0 | z_t)` at every pixel. The absorbing-state SUBS parameterization reduces the discrete-time variational objective to a weighted masked cross-entropy: only masked locations contribute, weighted by the probability that a masked location is revealed in the reverse transition,

`(alpha_{t-1} - alpha_t) / (1 - alpha_t)`.

The loss below averages the timestep terms by sampling `t` uniformly. The live Plotly loss chart updates in place.

In [5]:
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 2e-4
PLOT_EVERY = 100

train_loader = DataLoader(
    TensorDataset(train_mazes), batch_size=BATCH_SIZE, shuffle=True, drop_last=True
)
val_loader = DataLoader(TensorDataset(val_mazes), batch_size=BATCH_SIZE, shuffle=False)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)


def masked_diffusion_loss(maze_batch):
    x0 = maze_batch.to(DEVICE)
    timesteps = torch.randint(1, DIFFUSION_STEPS + 1, (x0.shape[0],), device=DEVICE)
    z_t = q_sample(x0, timesteps)
    logits_x0 = model(one_hot_states(z_t), timesteps)
    pixel_cross_entropy = F.cross_entropy(logits_x0, x0, reduction="none")
    masked = z_t.eq(MASK_TOKEN)
    masked_cross_entropy = (pixel_cross_entropy * masked).flatten(1).sum(dim=1)

    alpha_previous = alpha[timesteps - 1]
    alpha_current = alpha[timesteps]
    reveal_probability = (alpha_previous - alpha_current) / (1.0 - alpha_current)
    return (reveal_probability * masked_cross_entropy).mean()


@torch.no_grad()
def validation_loss():
    model.eval()
    values = [masked_diffusion_loss(batch).item() for (batch,) in val_loader]
    model.train()
    return float(np.mean(values))


model.train()
train_history, val_history = [], []
loss_fig = go.Figure()
loss_fig.add_scatter(name="train", mode="lines")
loss_fig.add_scatter(name="validation", mode="lines+markers")
loss_fig.update_layout(
    title="MDLM masked diffusion loss",
    xaxis_title="Optimization step",
    yaxis_title="Weighted masked cross-entropy per maze",
)
global_step = 0
for epoch in range(EPOCHS):
    for (batch,) in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = masked_diffusion_loss(batch)
        loss.backward()
        optimizer.step()
        train_history.append(loss.item())
        global_step += 1
        if global_step % PLOT_EVERY == 0:
            loss_fig.data[0].x = list(range(1, len(train_history) + 1))
            loss_fig.data[0].y = train_history
            loss_fig.data[1].x = [step for step, _ in val_history]
            loss_fig.data[1].y = [value for _, value in val_history]
            loss_fig.update_layout(
                title=f"MDLM training | epoch {epoch + 1}/{EPOCHS} | step {global_step}"
            )
            clear_output(wait=True)
            display(loss_fig)
    epoch_train = float(np.mean(train_history[-len(train_loader) :]))
    epoch_validation = validation_loss()
    val_history.append((global_step, epoch_validation))
    loss_fig.data[0].x = list(range(1, len(train_history) + 1))
    loss_fig.data[0].y = train_history
    loss_fig.data[1].x = [step for step, _ in val_history]
    loss_fig.data[1].y = [value for _, value in val_history]
    loss_fig.update_layout(
        title=f"MDLM training | epoch {epoch + 1}/{EPOCHS} | step {global_step}"
    )
    clear_output(wait=True)
    display(loss_fig)
    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS} | train={epoch_train:.4f} | val={epoch_validation:.4f}"
    )

print(f"Training complete | train={epoch_train:.4f} | val={epoch_validation:.4f}")

Epoch 30/30 | train=1.5542 | val=1.5636
Training complete | train=1.5542 | val=1.5636


## 5. Save the Model

In [6]:
ARTIFACTS = PROJECT_ROOT / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
checkpoint_path = ARTIFACTS / "maze_mdlm_subs.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": {
            "num_classes": NUM_CLASSES,
            "mask_token": MASK_TOKEN,
            "diffusion_steps": DIFFUSION_STEPS,
            "base_channels": BASE_CHANNELS,
            "time_dim": TIME_DIM,
            "retention_schedule": "linear",
            "parameterization": "subs_x0_prediction",
        },
    },
    checkpoint_path,
)
print(f"Saved: {checkpoint_path}")

Saved: c:\Users\shich\Src\discrete_diffusion\artifacts\maze_mdlm_subs.pt


## 6. Reverse Sampling and Solvability

Start from a fully masked grid. At reverse step `t`, keep already revealed pixels fixed. For each masked pixel, reveal it with probability `(alpha_{t-1} - alpha_t) / (1 - alpha_t)`; when revealed, sample its wall/passage class from the model's predicted `p_theta(x_0 | z_t)`. At `t=1`, the reveal probability is one, so every remaining mask is filled.

In [7]:
N_METRIC_SAMPLES = 128
N_TO_SHOW = 4
REVERSE_PREVIEW_TIMESTEPS = [100, 90, 80, 70, 60, 50, 40, 30, 20, 10, 0]


@torch.no_grad()
def sample_mdlm(model, n_samples, capture_timesteps=()):
    model.eval()
    z_t = torch.full((n_samples, 32, 32), MASK_TOKEN, device=DEVICE, dtype=torch.long)
    captures = {DIFFUSION_STEPS: z_t.detach().clone()}
    capture_timesteps = set(capture_timesteps)

    for t in reversed(range(1, DIFFUSION_STEPS + 1)):
        timesteps = torch.full((n_samples,), t, device=DEVICE, dtype=torch.long)
        logits_x0 = model(one_hot_states(z_t), timesteps)
        probabilities_x0 = logits_x0.softmax(dim=1)
        sampled_classes = torch.multinomial(
            probabilities_x0.permute(0, 2, 3, 1).reshape(-1, NUM_CLASSES), 1
        ).reshape(n_samples, 32, 32)

        alpha_previous = alpha[t - 1]
        alpha_current = alpha[t]
        reveal_probability = (alpha_previous - alpha_current) / (1.0 - alpha_current)
        reveal = z_t.eq(MASK_TOKEN) & (
            torch.rand(z_t.shape, device=DEVICE) < reveal_probability
        )
        z_t = torch.where(reveal, sampled_classes, z_t)
        if (t - 1) in capture_timesteps:
            captures[t - 1] = z_t.detach().clone()

    return z_t, captures


samples, reverse_snapshots = sample_mdlm(
    model, N_METRIC_SAMPLES, REVERSE_PREVIEW_TIMESTEPS
)
hard_samples = samples.cpu().numpy().astype(np.uint8)

fig = make_subplots(
    rows=N_TO_SHOW,
    cols=len(REVERSE_PREVIEW_TIMESTEPS),
    subplot_titles=[
        f"t={t}" for _ in range(N_TO_SHOW) for t in REVERSE_PREVIEW_TIMESTEPS
    ],
)
for row in range(N_TO_SHOW):
    for col, timestep in enumerate(REVERSE_PREVIEW_TIMESTEPS, start=1):
        fig.add_trace(
            go.Heatmap(
                z=reverse_snapshots[timestep][row].cpu().numpy(),
                zmin=0,
                zmax=2,
                colorscale=mask_colors,
                showscale=False,
            ),
            row=row + 1,
            col=col,
        )
fig.update_layout(
    title="Reverse MDLM sampling: black=wall, white=passage, gray=[MASK]",
    height=850,
    width=1900,
)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False, autorange="reversed")
fig.show()

solved = [solvability_rate([maze]) for maze in hard_samples]
print(f"Solvable samples: {sum(solved)}/{len(solved)} ({np.mean(solved):.1%})")
hard_fig = make_subplots(
    rows=1,
    cols=N_TO_SHOW,
    subplot_titles=[
        f"Sample {i + 1}: {'solvable' if solved[i] else 'unsolvable'}"
        for i in range(N_TO_SHOW)
    ],
)
for i in range(N_TO_SHOW):
    hard_fig.add_trace(
        go.Heatmap(
            z=hard_samples[i],
            zmin=0,
            zmax=1,
            colorscale=[[0, "black"], [1, "white"]],
            showscale=False,
        ),
        row=1,
        col=i + 1,
    )
hard_fig.update_layout(title="Final MDLM maze samples", height=390, width=900)
hard_fig.update_xaxes(showticklabels=False)
hard_fig.update_yaxes(showticklabels=False, autorange="reversed")
hard_fig.show()

Solvable samples: 36.0/128 (28.1%)
